# Lecture 1 — Practical: *Meeting Your Transcriptomic Dataset*
### Practical Machine Learning for Transcriptomics in Cancer Research

**No modelling today.** The whole point of this session is to *know your data before you trust it*.
You will load a real transcriptomic dataset, audit its clinical metadata, define a clean label,
measure class balance, run PCA, discover a batch effect, and build a defensible
train / validation / test split.

Adopt the mindset of an analyst writing the **"data sanity"** section of a methods paper — not
someone racing to a result. Every decision you make should be **written down** as you go.

---

#### The data (real cohorts)

| Cohort | Platform | Role | Source |
|---|---|---|---|
| **METABRIC** (~1,900 tumours) | Illumina HT-12 | primary cohort, long-term survival follow-up | cBioPortal `brca_metabric` |
| **GSE6532** (Loi et al.) | Affymetrix | second platform — combined in **deliberately** | NCBI GEO |

We predict **recurrence** (a relapse / DMFS-style **binary** endpoint). METABRIC has **no pCR** —
that's expected, and is itself a teaching point about letting the data choose the endpoint.

> **Why combine two cohorts on two platforms?** Because that is exactly where a real **batch
> effect** is born — and Section 4 is about discovering it. This is a *constructed teaching
> example*, not how you would assemble a genuine validation set.

> **Network note.** These cells download real data from cBioPortal and GEO. You need internet
> access. Downloads are cached to the lesson's `practical/task/datasets/` folder (resolved
> automatically), so you only fetch once. The data files are **git-ignored** — they are not
> committed to the repository; re-running this notebook re-fetches them on demand.


## Section 0 — Setup & framing  *(≈10 min)*

First, install/import what we need and create a cache directory.


In [ ]:
# Core scientific stack (usually preinstalled). GEOparse fetches GSE6532 from GEO.
# Run once; safe to re-run.
import sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import GEOparse  # noqa
except ImportError:
    _pip("GEOparse")

import os, tarfile, io, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
np.random.seed(0)          # reproducibility for the splits later

def _resolve_data_dir():
    """Find the lesson's practical/task/datasets/ folder regardless of where Jupyter
    was launched (the student notebook runs from notebooks/, the solution from
    solutions/lesson01_biological_question/notebooks/ — both should share ONE cache).

    Repo layout this targets:
      lessons/lesson01_biological_question/practical/task/
        ├── notebooks/   <- student notebook usually runs from here
        └── datasets/    <- data is cached here (git-ignored, kept via .gitkeep)
    """
    here = Path.cwd()
    # 1) PRIORITY: walk upward to the repo root, then locate the lesson-01 datasets anchor
    #    by glob (robust to the exact topic suffix), so the student and solution notebooks
    #    share the same cache.
    for parent in [here, *here.parents]:
        lessons = parent / "lessons"
        if lessons.is_dir():
            for cand in sorted(lessons.glob("lesson01_*/practical/task/datasets")):
                cand.mkdir(parents=True, exist_ok=True)
                return cand
            # lessons/ exists but no datasets dir yet -> create the canonical one if the
            # lesson folder is present
            for cand in sorted(lessons.glob("lesson01_*/practical/task")):
                d = cand / "datasets"; d.mkdir(parents=True, exist_ok=True)
                return d
    # 2) Otherwise, try datasets next to / beside the notebook.
    for c in [here.parent / "datasets", here / "datasets"]:
        if c.parent.exists():
            c.mkdir(parents=True, exist_ok=True)
            return c
    # 3) Last resort.
    fallback = here / "datasets"
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback

DATA_DIR = str(_resolve_data_dir())
print("Setup complete. Data will be cached in:")
print("  ", os.path.abspath(DATA_DIR))
print("(Large downloads are git-ignored; the folder is kept via .gitkeep.)")

### The prediction task (write it in your own words)

Before touching any code, state the prediction task **precisely**. A well-posed task names four
things: the **input**, the **output (label)**, **when** each is measured, and **on whom**.

> **Exercise 0.1 — edit the markdown below and commit your task statement.**


**My task statement:**

*(TODO — replace this with your own. Name the input, the output/label, when each is measured,
and on whom. Two or three sentences is plenty.)*

- **Input (X):** …
- **Output (y):** …
- **When:** …
- **On whom:** …


---
## Data loaders (shared infrastructure)

The two functions below fetch the **real** cohorts and cache them. Read them, but you don't need
to edit them — the teaching happens *after* the data is loaded. Each returns a tidy
`(expression, clinical)` pair where **expression is genes × samples** (the bioinformatics
convention — you will fix the orientation yourself in Section 1).


In [ ]:
# Data loaders. METABRIC comes from the cBioPortal datahub (now served as individual
# files via GitHub LFS); GSE6532 comes from NCBI GEO over HTTPS. Run once; cached after.
CBIO_BASE = ("https://media.githubusercontent.com/media/cBioPortal/datahub/"
             "master/public/brca_metabric")
CBIO_FILES = {
    "expr":    "data_mrna_illumina_microarray.txt",   # large (~660 MB)
    "patient": "data_clinical_patient.txt",
    "sample":  "data_clinical_sample.txt",
}

def _download(url, dest):
    """Stream a URL to dest, skipping the download if a non-empty cache exists."""
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return dest
    print(f"Downloading {os.path.basename(dest)} ...")
    r = requests.get(url, stream=True, timeout=300, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    with open(dest, "wb") as fh:
        for chunk in r.iter_content(chunk_size=1 << 20):
            fh.write(chunk)
    return dest

def _read_cbio_clinical(path):
    """cBioPortal clinical files carry 4 '#'-commented header lines above the real header."""
    return pd.read_csv(path, sep="\t", comment="#", low_memory=False)

def load_metabric(data_dir=DATA_DIR):
    """Download & parse METABRIC (brca_metabric) from the cBioPortal datahub.

    The expression file is large (~660 MB); the first run takes a few minutes, then caches.

    Returns
    -------
    expr : DataFrame  (genes x samples)  -- Illumina HT-12 microarray, log-intensity
    clin : DataFrame  (samples x clinical fields)
    """
    paths = {k: _download(f"{CBIO_BASE}/{fn}", os.path.join(data_dir, fn))
             for k, fn in CBIO_FILES.items()}
    expr = pd.read_csv(paths["expr"], sep="\t", low_memory=False)
    expr = expr.drop(columns=[c for c in ["Entrez_Gene_Id"] if c in expr.columns])
    expr = expr.dropna(subset=["Hugo_Symbol"]).set_index("Hugo_Symbol")
    expr = expr[~expr.index.duplicated(keep="first")]      # one row per gene symbol
    pat  = _read_cbio_clinical(paths["patient"])
    smp  = _read_cbio_clinical(paths["sample"])
    clin = pat.merge(smp, on="PATIENT_ID", how="inner", suffixes=("", "_smp"))
    if "SAMPLE_ID" in clin.columns:
        clin = clin.set_index("SAMPLE_ID")
    return expr, clin


GSE_SOFT_URL = ("https://ftp.ncbi.nlm.nih.gov/geo/series/GSE6nnn/GSE6532/"
                "soft/GSE6532_family.soft.gz")

def load_gse6532(data_dir=DATA_DIR):
    """Download & parse GSE6532 (Loi et al.) from GEO.

    We fetch the family SOFT file over HTTPS (more reliable than GEOparse's default FTP),
    restrict to the single most-represented Affymetrix platform, and map its probes to
    gene symbols via the platform (GPL) annotation. The result is a symbol-indexed matrix
    that shares thousands of genes with METABRIC -- the common feature space the PCA
    batch demo needs.

    Returns
    -------
    expr : DataFrame  (genes x samples)  -- Affymetrix, symbol-collapsed (mean over probes)
    clin : DataFrame  (samples x clinical fields)
    """
    import GEOparse, collections
    soft = _download(GSE_SOFT_URL, os.path.join(data_dir, "GSE6532_family.soft.gz"))
    gse  = GEOparse.get_GEO(filepath=soft, silent=True)

    plat_of = {name: gsm.metadata.get("platform_id", ["?"])[0]
               for name, gsm in gse.gsms.items()}
    dom = collections.Counter(plat_of.values()).most_common(1)[0][0]   # dominant platform

    # probe -> gene symbol from the platform annotation
    gpl = gse.gpls[dom].table
    sym_col = next((c for c in gpl.columns
                    if c.lower() in ("gene symbol", "gene_symbol", "symbol")), None)
    pmap = gpl.set_index("ID")[sym_col].dropna().astype(str)
    pmap = pmap[pmap.str.len() > 0]

    cols, meta_rows = {}, {}
    for name, gsm in gse.gsms.items():
        if plat_of[name] != dom:
            continue
        tbl = gsm.table
        if tbl is None or "VALUE" not in tbl.columns:
            continue
        cols[name] = pd.Series(tbl["VALUE"].values, index=tbl["ID_REF"].astype(str).values)
        ch = gsm.metadata.get("characteristics_ch1", [])
        row = {"title": "; ".join(gsm.metadata.get("title", []))}
        for item in ch:
            if ":" in item:
                k, v = item.split(":", 1)
                row[k.strip().lower()] = v.strip()
        meta_rows[name] = row

    expr = pd.DataFrame(cols)                        # probes x samples
    expr = expr[expr.index.isin(pmap.index)]
    expr.index = pmap.loc[expr.index].values         # probes -> gene symbols
    expr = expr.groupby(level=0).mean()              # collapse probes per gene
    clin = pd.DataFrame(meta_rows).T
    return expr, clin

print("Loaders defined: load_metabric(), load_gse6532()")

In [ ]:
# Fetch both cohorts (cached after first run).
metabric_expr, metabric_clin = load_metabric()
print("METABRIC expression (genes x samples):", metabric_expr.shape)
print("METABRIC clinical:", metabric_clin.shape)

gse_expr, gse_clin = load_gse6532()
print("GSE6532 expression (probes x samples):", gse_expr.shape)
print("GSE6532 clinical:", gse_clin.shape)

---
## Section 1 — Loading & orienting the data  *(≈20 min)*

The loaders gave you expression as **genes × samples** (the bioinformatics convention). The ML
convention is **samples × genes** (scikit-learn expects rows = examples). Transposition errors are
a classic, embarrassing bug — so you will determine the orientation *from evidence*, then assert it.


> **Exercise 1.1 — determine orientation from evidence, then assert it.**
>
> Don't just trust the docstring. Look at the shapes and the index/columns and *reason* about which
> axis is samples and which is genes. Then transpose METABRIC to **samples × genes** and assert it.
>
> *Hints:*
> - There are ~20,000 genes but only ~1,900 samples — the long axis is genes.
> - Gene names look like `ESR1`, `MKI67`; sample IDs look like `MB-0001`.
> - After transposing, `X.shape[0]` should be the number of samples (the smaller number).


In [ ]:
# TODO 1.1
# (a) Inspect metabric_expr: what do the index and columns look like? what is the shape?
print("rows (index) sample:", list(metabric_expr.index[:3]), "...")
print("cols sample:", list(metabric_expr.columns[:3]), "...")
print("shape:", metabric_expr.shape)

# (b) Decide which axis is samples and which is genes, and transpose to samples x genes.
#     X_metabric = ...

# (c) ASSERT your orientation (e.g. genes should outnumber samples; 'ESR1' among columns).
#     Also report value range and number of missing values.
# X_metabric = ...
# assert ...


> **Exercise 1.2 — join expression to clinical metadata, and investigate failures.**
>
> Join `X_metabric` (indexed by sample ID) to `metabric_clin` (also indexed by `SAMPLE_ID`).
> Report how many samples matched, and **look at** the ones that didn't — don't silently drop them.
>
> *Hint:* `X_metabric.index` vs `metabric_clin.index`; use set operations to see the mismatch.


In [ ]:
# TODO 1.2
# Compare the sample IDs in X_metabric.index vs metabric_clin.index.
# Report: how many matched? how many are expression-only / clinical-only?
# Then keep only matched samples, aligned in the same order, as X_metabric and clin.
#
# expr_ids = set(X_metabric.index); clin_ids = set(metabric_clin.index)
# ...


> **Discussion 1 (write a sentence).** What real-world events cause samples to fail a join
> (relabelled IDs, withdrawn consent, QC failures, version drift)? Why is *silently* dropping
> unmatched samples dangerous for the conclusions you'll later draw?


---
## Section 2 — Auditing the clinical metadata and the label  *(≈25 min)*

METABRIC has **no pCR**. It has long-term survival follow-up, so our label is **binary recurrence**.
You must turn messy survival fields into a clean target — and **record every decision**.


First, filter to the course cohort — **HR+/HER2−** — to match the clinical question.

> **Exercise 2.0 — filter to HR+/HER2−** using the receptor-status columns, and report how many
> tumours remain.
>
> *Hint:* METABRIC clinical columns include `ER_STATUS`, `PR_STATUS`, `HER2_STATUS`
> (values like `Positive`/`Negative`). HR+ = ER **or** PR positive; HER2− = HER2 negative.


In [ ]:
# TODO 2.0
# Look at ER_STATUS / PR_STATUS / HER2_STATUS, then build a boolean mask for HR+/HER2-.
# HR+ = ER positive OR PR positive ;  HER2- = HER2 negative.
# Apply the mask to BOTH clin and X_metabric, and report how many remain.
#
# for c in ["ER_STATUS", "PR_STATUS", "HER2_STATUS"]:
#     print(c, clin[c].value_counts(dropna=False))
# mask = ...


> **Exercise 2.1 — define a binary recurrence label, with an explicit censoring policy.**
>
> METABRIC exposes relapse-free survival fields (commonly `RFS_STATUS` like
> `"0:Not Recurred"` / `"1:Recurred"`, and `RFS_MONTHS`). Define:
>
> `y = 1` if the patient **recurred within the horizon** (e.g. 60 months);
> `y = 0` if the patient was **followed at least the horizon with no recurrence**;
> **exclude** patients censored *before* the horizon (we cannot know their outcome).
>
> Report how many are positive, negative, and excluded — and **write down** your horizon and
> censoring rule. *A censored patient is **not** automatically a non-event.*


In [ ]:
# TODO 2.1
# Pick a HORIZON (months). Find the relapse status & months columns (RFS_* or DFS_*).
# Build y: 1 = recurred within horizon; 0 = event-free through horizon;
#          NaN = censored before horizon (EXCLUDE -- unknown outcome).
# Apply the keep-mask to clin, X_metabric, y. Report positives / negatives / excluded.
#
# HORIZON = ...
# status_col = ...; months_col = ...
# recurred = ...; months = ...
# y = pd.Series(index=clin.index, dtype="float")
# ...


> **Exercise 2.2 — name a leaky field.** Identify at least one metadata column that must **not**
> be used as a feature because it would leak — anything measured *at or after* the outcome.
>
> *Hint:* survival/vital-status columns, the relapse fields themselves, or treatment given *in
> response to* progression. Write one sentence explaining why.


In [ ]:
# TODO 2.2
# List metadata columns that would leak the outcome if used as features, and say why in a comment.
# leaky = [c for c in clin.columns if ...]
# print(leaky)


> **Discussion 2 (write a few sentences).** The published model uses RFS for development and DMFS
> for external validation, arguing DMFS is the more faithful long-term endpoint. Why does the choice
> (any relapse vs distant-only) change *both* the event count and the clinical meaning? And: pCR
> was simply **unavailable** here — why is *"use the endpoint your data supports"* a defensible
> scientific decision rather than a compromise to hide?


---
## Section 3 — Class balance  *(≈15 min)*

Recurrence events are a **minority**. That has consequences for metrics (accuracy will mislead)
and for splitting (you must stratify).


> **Exercise 3.1 — quantify and visualise class balance**, and state the **smallest class count**
> (the number of recurrence events). Plot the two counts as a bar chart (Figure matches the deck's
> class-balance figure).
>
> **Exercise 3.2 (writing).** Before you test it in Section 5: predict, in a sentence, how a naive
> 80/20 random split could distribute the minority class badly. What is the worst case?


In [ ]:
# TODO 3.1 / 3.2
# Compute counts of y==0 and y==1, the proportions, and the smallest class count.
# Make a 2-bar bar chart. Then in a comment, note the accuracy of an 'always no-relapse' model.
# Finally answer Exercise 3.2 in the markdown cell below.
#
# counts = y.value_counts().sort_index()
# ...


**My Section 3.2 prediction (worst case of a naive random split):**

*(TODO in the student notebook — write your prediction here before Section 5.)*


---
## Section 4 — PCA and batch effects  *(≈40 min — a core section)*

Now we **combine METABRIC with GSE6532** on their shared genes. The two platforms (Illumina vs
Affymetrix) create a strong technical signal. PCA will let us *see* it.

First, build the combined matrix and a `batch` label. (This cell is shared infrastructure — read it,
then the exercises follow.)


In [ ]:
# GSE6532 is now symbol-indexed (genes x samples); orient to samples x genes and
# intersect with METABRIC on shared gene symbols -> the common feature space for PCA.
gse_X = gse_expr.T.copy()                       # samples x genes
common_genes = [g for g in X_metabric.columns if g in gse_X.columns]
print("common features (shared gene symbols) used for PCA:", len(common_genes))
assert len(common_genes) > 1000, "Expected thousands of shared genes after probe->symbol mapping."

# Build combined matrix on the shared genes. We deliberately DO NOT batch-correct here,
# so the cross-platform batch effect stays visible in the PCA.
A = X_metabric[common_genes].copy()
B = gse_X[common_genes].copy()
combined = pd.concat([A, B], axis=0)
batch = pd.Series(["METABRIC/Illumina"]*len(A) + ["GSE6532/Affymetrix"]*len(B),
                  index=combined.index, name="batch")
print("combined matrix (samples x genes):", combined.shape)
print(batch.value_counts().to_dict())

> **Exercise 4.1 — run PCA and produce the two-colouring plot.**
>
> Standardise the combined features (for visualisation only — flag it as *exploratory, not a
> modelling step*), run PCA to 2 components, and make **two scatter plots of the same points**:
> one coloured by `batch`, one coloured by the recurrence label (for METABRIC samples; GSE6532
> can be shown as "unlabelled"). Describe what dominates PC1.
>
> *Hints:* `from sklearn.preprocessing import StandardScaler`, `from sklearn.decomposition import PCA`.
> Drop genes with any NaN before PCA (`combined.dropna(axis=1)`).


In [ ]:
# TODO 4.1
# Standardise M = combined.dropna(axis=1) (exploratory only!), PCA -> 2 comps, print variance.
# Then TWO scatter plots of the SAME points: coloured by batch, and by recurrence label.
# (Align the recurrence label to the combined index; GSE6532 samples are "unlabelled".)
#
# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import PCA
# ...


> **Exercise 4.2 — is batch confounded with outcome?** Quantify it: cross-tabulate `batch`
> against the recurrence label (METABRIC samples only, since GSE6532 is unlabelled here). State a
> verdict with evidence.
>
> **Exercise 4.3 (reasoning, write it).** If batch and outcome were *perfectly* confounded, explain
> why no correction method (e.g. ComBat) could rescue the analysis.


In [ ]:
# TODO 4.2 / 4.3
# Cross-tabulate batch vs recurrence label (METABRIC samples only) and write a verdict.
# Then answer 4.3 in the markdown cell below.
# tab = pd.crosstab(...); print(tab)


**My Section 4.3 answer (perfect confounding):**

*(TODO in the student notebook.)*


> **Discussion 4.** PCA shows structure but not its *cause*. What extra metadata would you request
> from the data generators to be sure PC1 is technical (platform) and not a genuine biological
> difference between the cohorts' patient populations?
>
> **Common misconception to retire:** that normalisation already "removed" batch, or that PCA is a
> correction step. PCA only *reveals*.


---
## Section 5 — Designing the splits  *(≈40 min — a core section)*

We now build splits **on the METABRIC labelled cohort** (the part with a usable label). A good split
is simultaneously **patient-level**, **stratified** by the label, and **batch-aware**.

> *Note on this data:* METABRIC is ~one-sample-per-patient, so true duplication isn't present —
> implement patient-level grouping anyway, as a discipline. The live constraint here is
> stratification of the minority class.


> **Exercise 5.1 — build a NAIVE random split first**, then measure how the minority class
> actually landed. Compare to your Section 3.2 prediction.
>
> *Hint:* `from sklearn.model_selection import train_test_split` with **no** `stratify`. Use a couple
> of different `random_state` values and watch the test-set positive rate wobble.


In [ ]:
# TODO 5.1
# Build a NAIVE 80/20 split (no stratify) for a few random_state values and print the test-set
# positive rate each time. Note how much it moves. Compare to your Section 3.2 prediction.
#
# from sklearn.model_selection import train_test_split
# Xlab = X_metabric.loc[y.index]
# for rs in [0,1,2,3]: ...


> **Exercise 5.2 — build the proper split and verify integrity.**
>
> Make a **patient-level, stratified** train/validation/test split (e.g. 60/20/20). Then run the
> integrity checks: (i) no patient appears in two splits; (ii) class proportions are preserved in
> each split; (iii) report the batch composition (here all METABRIC, but check the habit).
>
> *Hints:* `StratifiedGroupKFold` (groups = patient ID) or a two-step `train_test_split` with
> `stratify=y`. For METABRIC the patient ID is `PATIENT_ID` in `clin`.


In [ ]:
# TODO 5.2
# Build a patient-level, stratified 60/20/20 split.
# Verify: (i) zero patient overlap across splits; (ii) class proportions preserved;
#         (iii) batch composition per split.
#
# groups = clin.loc[y.index, "PATIENT_ID"]
# tr, tmp = train_test_split(..., stratify=y...) ; va, te = train_test_split(...)
# ... assert no overlap ; print proportions ; print batch per split


> **Exercise 5.3 — the leakage hunt (the payoff).**
>
> In writing: where in a typical pipeline would you normalise and feature-select, and **why must
> both happen *after* and *inside* the split**? Then point to the exact line in the naive workflow
> (below) where leakage occurs.


In [ ]:
# A DELIBERATELY BROKEN workflow -- find the leak (do NOT copy this into real work):
#
#   1  scaler = StandardScaler().fit(Xlab.values)          # <-- uses ALL samples (train+test)
#   2  Xall   = scaler.transform(Xlab.values)
#   3  top    = pick_top_genes_by_association(Xall, y)     # <-- selects on ALL samples incl. test
#   4  tr, te = train_test_split(Xall[:, top], y)          # <-- split happens LAST
#   5  model.fit(Xall[tr], y[tr]); model.score(Xall[te])   # test set already influenced steps 1 & 3
#
# Lines 1 and 3 are the leaks: scaling statistics and feature ranking both saw the test labels/values
# BEFORE the split. The fix: split FIRST, then fit the scaler and select features on TRAIN ONLY,
# and apply those fixed choices to val/test.
print("Leak locations: line 1 (normalisation over train+test) and line 3 (feature selection over "
      "train+test). Both must move to AFTER the split, fit on TRAIN only.")

**My Section 5.3 answer:**

*(TODO in the student notebook — explain where normalisation & feature selection belong, and name
the leaking lines above.)*


> **Discussion 5.** Why is an independent **external cohort** worth more than any internal split?
> What would you look for in a candidate validation cohort — platform, patient population, processing,
> endpoint definition?


---
## Section 6 — Reflection: the data-readiness memo  *(≈15 min — main assessable artefact)*

Write a **200–300 word "data readiness" memo**. Is this dataset fit for building a *trustworthy*
recurrence biomarker? Cover its risks — sample size, class imbalance, batch confounding across the
two platforms, label/censoring ambiguity — and say what you would fix or request before any modelling.
This is your honest analyst's verdict.


**Data-readiness memo (200–300 words):**

*(TODO — write your verdict here.)*


---
### Deliverables checklist

- [ ] Written prediction-task statement (Section 0)
- [ ] Confirmed matrix orientation + clean joined table (Section 1)
- [ ] Clean binary label vector + label-decisions log incl. censoring policy (Section 2)
- [ ] Class-balance figure + interpretation (Section 3)
- [ ] Two-colouring PCA plots + batch/confounding verdict (Section 4)
- [ ] Patient-level, stratified, batch-aware split + integrity-check table (Section 5)
- [ ] Naive-vs-proper split comparison + identified leakage point (Section 5)
- [ ] Final data-readiness memo (Section 6)

> **Remember the message of the whole course:** *building a trustworthy predictive biomarker is more
> important than choosing a sophisticated algorithm.* Everything you did today is that rigour, before
> a single model is trained.
